# Dehydrated xylem vessels: human annotation vs Cellpose-SAM

For reviewer comment 2 (independent check of the instance reference). For each selected slice:

1. **Human annotation**: dehydrated xylem vessels (class 7) from the manually drawn semantic mask
   (`data/annotations/{slice}_mask.npy`) → regions and their polygons.
2. **Cellpose-SAM**: instance masks (`data/instance_masks/{slice}_masks.tif`) → the Cellpose-SAM cells that lie in the
   dehydrated regions, using a slightly **relaxed** (dilated) region so that cells whose boundary extends a little beyond
   the hand-drawn outline are not cut off → their polygons.
3. **Comparison** of the two polygon sets: area, pixel IoU / Dice, precision / recall, boundary distance (mean, 95th
   percentile), and per-region agreement.

Note: in the pipeline (`2get_class_mask.ipynb`) a Cellpose-SAM cell gets class 7 because it overlaps the human region,
so *which* cells are dehydrated xylem is not independent of the human mask. What this notebook measures independently is
the **delineation**: how well the Cellpose-SAM cell outlines reproduce the human-drawn vessel outlines.

In [ ]:
# =============================================================================
# Config
# =============================================================================
from pathlib import Path

DATA_DIR = Path("./data")          # on NERSC: /pscratch/sd/x/xchong/sam3_finetune/seg_annotation_pipeline2_7classes/data
SLICES = [
    "20260212_133951_petiole30_00100",
    "20260212_174538_petiole36_00100",
    "20260213_145242_petiole47_00100",
]
DEHYD = 7                          # class id of dehydrated xylem vessels in the semantic mask

RELAX_PX = 5                       # dilation of the human region before selecting Cellpose cells (relaxed boundary)
MIN_CELL_FRACTION = 0.5            # a Cellpose cell is kept if >= this fraction of its area lies in the relaxed region
MIN_REGION_AREA = 20               # px; ignore tiny human fragments
MATCH_IOU = 0.5                    # region-level match threshold

OUT_DIR = Path("./dehydrated_xylem_human_vs_cellpose")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from scipy import ndimage as ndi

def find(d, stem, suffixes):
    for s in suffixes:
        p = Path(d) / f"{stem}{s}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{stem}{suffixes} not in {d}")

def load_image(stem):
    a = np.array(Image.open(find(DATA_DIR / "images", stem, (".tiff", ".tif")))).astype(np.float32)
    lo, hi = np.percentile(a, (0.5, 99.5))
    return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)

def load_semantic(stem):
    return np.load(find(DATA_DIR / "annotations", stem, ("_mask.npy",)))

def load_instances(stem):
    a = np.array(Image.open(find(DATA_DIR / "instance_masks", stem, ("_masks.tif", "_masks.tiff"))))
    return (a[..., 0] if a.ndim == 3 else a).astype(np.int64)

def disk(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))

def mask_to_polygons(mask):
    """Outer polygons of a binary mask (one per connected region), as lists of (x, y)."""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c[:, 0, :].tolist() for c in contours if len(c) >= 3]

def labels_to_polygons(lab):
    """One polygon per instance id."""
    polys = {}
    for obj_id, sl in enumerate(ndi.find_objects(lab), start=1):
        if sl is None:
            continue
        sub = (lab[sl] == obj_id).astype(np.uint8)
        cs, _ = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cs:
            c = max(cs, key=cv2.contourArea)[:, 0, :] + np.array([sl[1].start, sl[0].start])
            polys[obj_id] = c.tolist()
    return polys

def rasterize(polys, shape):
    m = np.zeros(shape, np.uint8)
    for p in polys:
        cv2.fillPoly(m, [np.asarray(p, np.int32)], 1)
    return m.astype(bool)

def boundary(mask):
    return mask & ~ndi.binary_erosion(mask, structure=np.ones((3, 3)))

def boundary_distances(a, b):
    """Distances from boundary pixels of a to the boundary of b and vice versa (px)."""
    ba, bb = boundary(a), boundary(b)
    if not ba.any() or not bb.any():
        return np.array([np.nan])
    da = ndi.distance_transform_edt(~bb)[ba]
    db = ndi.distance_transform_edt(~ba)[bb]
    return np.concatenate([da, db])

## Human regions, Cellpose-SAM cells, polygons and comparison

In [ ]:
rows, region_rows, cache = [], [], {}
for stem in SLICES:
    sem, inst = load_semantic(stem), load_instances(stem)
    assert sem.shape == inst.shape, f"{stem}: {sem.shape} vs {inst.shape}"

    # ---- 1. human: dehydrated xylem regions ----
    human = sem == DEHYD
    lab_h, n_h = ndi.label(human, structure=np.ones((3, 3)))
    sizes = np.bincount(lab_h.ravel()); keep = sizes >= MIN_REGION_AREA; keep[0] = False
    human = keep[lab_h]
    lab_h, n_h = ndi.label(human, structure=np.ones((3, 3)))
    human_polys = mask_to_polygons(human)

    # ---- 2. Cellpose-SAM cells inside the relaxed human region ----
    relaxed = cv2.dilate(human.astype(np.uint8), disk(RELAX_PX)).astype(bool) if RELAX_PX > 0 else human
    area = np.bincount(inst.ravel())
    inside = np.bincount(inst.ravel(), weights=relaxed.ravel(), minlength=len(area))
    frac = inside / np.maximum(area, 1)
    sel_ids = np.nonzero((frac >= MIN_CELL_FRACTION) & (np.arange(len(area)) > 0))[0]
    cp_lab = np.where(np.isin(inst, sel_ids), inst, 0)
    cp_polys = labels_to_polygons(cp_lab)

    # ---- 3. compare (polygons rasterised back to masks) ----
    H = rasterize(human_polys, human.shape)
    C = rasterize(cp_polys.values(), human.shape)
    inter, union = (H & C).sum(), (H | C).sum()
    bd = boundary_distances(H, C)
    rows.append(dict(slice=stem, human_regions=len(human_polys), cellpose_cells=len(cp_polys),
                     human_area_px=int(H.sum()), cellpose_area_px=int(C.sum()),
                     area_diff_pct=100 * (C.sum() - H.sum()) / max(H.sum(), 1),
                     IoU=inter / max(union, 1), Dice=2 * inter / max(H.sum() + C.sum(), 1),
                     precision=inter / max(C.sum(), 1), recall=inter / max(H.sum(), 1),
                     boundary_mean_px=np.nanmean(bd), boundary_p95_px=np.nanpercentile(bd, 95)))

    # per human region: IoU with the Cellpose cells assigned to it (majority overlap)
    lab_hp, _ = ndi.label(H, structure=np.ones((3, 3)))
    cell_region = {}
    for cid in cp_polys:
        rr = lab_hp[(cp_lab == cid) & (lab_hp > 0)]
        cell_region[cid] = np.bincount(rr).argmax() if rr.size else 0
    for rid in range(1, lab_hp.max() + 1):
        rmask = lab_hp == rid
        cids = [c for c, r in cell_region.items() if r == rid]
        cmask = np.isin(cp_lab, cids) if cids else np.zeros_like(rmask)
        i_, u_ = (rmask & cmask).sum(), (rmask | cmask).sum()
        region_rows.append(dict(slice=stem, region=rid, area_px=int(rmask.sum()), n_cellpose_cells=len(cids),
                                IoU=i_ / max(u_, 1)))
    cache[stem] = dict(H=H, C=C, human_polys=human_polys, cp_polys=cp_polys, relaxed=relaxed)

    # polygons for reuse
    json.dump({"slice": stem, "class": "Dehydrated Xylem vessels",
               "human_polygons": human_polys, "cellpose_polygons": {str(k): v for k, v in cp_polys.items()},
               "settings": dict(RELAX_PX=RELAX_PX, MIN_CELL_FRACTION=MIN_CELL_FRACTION)},
              open(OUT_DIR / f"{stem}_polygons.json", "w"))
    print(f"{stem}: {len(human_polys)} human regions, {len(cp_polys)} Cellpose-SAM cells, IoU {rows[-1]['IoU']:.3f}")

per_slice = pd.DataFrame(rows).set_index("slice")
per_region = pd.DataFrame(region_rows)

## Results

In [ ]:
# totals over the 3 slices (pixel metrics pooled; boundary distances averaged over slices)
Hs = sum(c["H"].sum() for c in cache.values()); Cs = sum(c["C"].sum() for c in cache.values())
I = sum((c["H"] & c["C"]).sum() for c in cache.values()); U = sum((c["H"] | c["C"]).sum() for c in cache.values())
total = dict(human_regions=per_slice["human_regions"].sum(), cellpose_cells=per_slice["cellpose_cells"].sum(),
             human_area_px=int(Hs), cellpose_area_px=int(Cs), area_diff_pct=100 * (Cs - Hs) / max(Hs, 1),
             IoU=I / max(U, 1), Dice=2 * I / max(Hs + Cs, 1), precision=I / max(Cs, 1), recall=I / max(Hs, 1),
             boundary_mean_px=per_slice["boundary_mean_px"].mean(), boundary_p95_px=per_slice["boundary_p95_px"].mean())
table = pd.concat([per_slice, pd.DataFrame([total], index=["all 3 slices"])])
pd.options.display.float_format = "{:.3f}".format
display(table)
table.to_csv(OUT_DIR / "comparison_per_slice.csv")

matched = (per_region["IoU"] >= MATCH_IOU).mean() if len(per_region) else np.nan
print(f"human regions with region-IoU >= {MATCH_IOU}: {100 * matched:.1f}% of {len(per_region)}; "
      f"median region IoU {per_region['IoU'].median():.3f}")
per_region.to_csv(OUT_DIR / "comparison_per_region.csv", index=False)

## Plots

In [ ]:
def crop_box(mask, pad=60):
    ys, xs = np.nonzero(mask)
    if not len(ys):
        return slice(None), slice(None)
    return (slice(max(ys.min() - pad, 0), ys.max() + pad), slice(max(xs.min() - pad, 0), xs.max() + pad))

def draw(ax, img, polys, color, title):
    ax.imshow(img, cmap="gray")
    for p in polys:
        p = np.asarray(p)
        ax.plot(np.r_[p[:, 0], p[0, 0]], np.r_[p[:, 1], p[0, 1]], color=color, lw=0.8)
    ax.set_title(title); ax.axis("off")

for stem in SLICES:
    c = cache[stem]; img = load_image(stem); r = per_slice.loc[stem]
    fig, axes = plt.subplots(1, 4, figsize=(22, 5.8))
    # full slice: human mask overlay
    ov = np.stack([img] * 3, -1); ov[c["H"]] = 0.5 * ov[c["H"]] + 0.5 * np.array([0, 0.78, 0.75])
    axes[0].imshow(ov); axes[0].set_title(f"Human annotation (class {DEHYD})"); axes[0].axis("off")
    sy, sx = crop_box(c["H"] | c["C"])
    off = np.array([sx.start or 0, sy.start or 0])
    sub = img[sy, sx]
    draw(axes[1], sub, [np.asarray(p) - off for p in c["human_polys"]], "cyan", f"Human polygons ({len(c['human_polys'])})")
    draw(axes[2], sub, [np.asarray(p) - off for p in c["cp_polys"].values()], "magenta",
         f"Cellpose-SAM polygons ({len(c['cp_polys'])} cells)")
    diff = np.zeros(sub.shape + (3,))
    Hc, Cc = c["H"][sy, sx], c["C"][sy, sx]
    diff[Hc & Cc] = (1, 1, 1); diff[Hc & ~Cc] = (0, 0.8, 0.8); diff[~Hc & Cc] = (1, 0, 1)
    axes[3].imshow(diff); axes[3].axis("off")
    axes[3].set_title(f"Difference  (white both, cyan human only, magenta Cellpose only)\n"
                      f"IoU {r.IoU:.3f}  Dice {r.Dice:.3f}  boundary {r.boundary_mean_px:.1f} px")
    fig.suptitle(stem); plt.tight_layout()
    plt.savefig(OUT_DIR / f"{stem}_human_vs_cellpose.png", dpi=150); plt.show()